# 02 — EDA (Data Understanding)

**Fase 2 — CRISP-DM: Data Understanding** · Cubre las Tareas 1–5 del enunciado (Criterios C1 + C2, 50% de la nota).

Este notebook trabaja sobre el dataset **crudo, sin modificar** (`data/raw/act_liver_disease.csv`, inmutable). No se imputa, no se elimina ningún registro y no se entrena ningún modelo — el objetivo es entender estructura, calidad, distribuciones y relaciones entre variables. Las decisiones de tratamiento (imputación, escalado, outliers) se toman en `03_preprocessing.ipynb`.

Ver `docs/adr/0003-sin-holdout-en-fases-0-3.md` para la decisión de no apartar ningún *holdout* antes de este análisis.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

from src.config import (
    AGE_ADULT_MIN,
    AGE_BAND_COLORS,
    AGE_BANDS,
    AGE_TOP_CODE,
    ALT_ULN_BY_SEX,
    ALT_ULN_UNISEX,
    CORR_THRESHOLD,
    DE_RITIS_ALCOHOLIC,
    DE_RITIS_VIRAL,
    DPI,
    GRID_COLOR,
    INK_MUTED,
    INK_PRIMARY,
    INK_SECONDARY,
    IQR_MULTIPLIER,
    NUMERIC_COLS,
    ORDINAL_RAMP_4,
    SEX_COLORS,
)
from src.utils import (
    alt_threshold_comparison,
    de_ritis_ratio,
    flag_biochemical_violations,
    load_raw_data,
    save_figure,
    whipple_index,
    whipple_label,
)

sns.set_theme(style="whitegrid")

df = load_raw_data()
df.shape

(583, 11)

## T1 — Estructura del dataset

¿Cuál es la estructura del dataset? Formato, tamaño, cantidad de variables y tipo de datos.

In [2]:
print("Formato: CSV delimitado por comas")
print("Shape (filas, columnas):", df.shape)
print("\nTipos de datos:")
print(df.dtypes)
print(f"\nMemoria en uso: {df.memory_usage(deep=True).sum() / 1024:.1f} KB")
df.head(3)

Formato: CSV delimitado por comas
Shape (filas, columnas): (583, 11)

Tipos de datos:
Age            int64
Gender           str
TB           float64
DB           float64
Alkphos        int64
Sgpt           int64
Sgot           int64
TP           float64
ALB          float64
A/G Ratio    float64
Selector       int64
dtype: object

Memoria en uso: 76.1 KB


,Age,Gender,TB,DB,Alkphos,Sgpt,Sgot,TP,ALB,A/G Ratio,Selector
0,65,Female,0.7,0.1,187,16,18,6.8,3.3,0.90,1
1,62,Male,10.9,5.5,699,64,100,7.5,3.2,0.74,1
2,62,Male,7.3,4.1,490,60,68,7.0,3.3,0.89,1


**Interpretación.** El dataset tiene **583 filas × 11 columnas** (≈76 KB en memoria) en formato CSV delimitado por comas. De las 11 columnas: **9 son numéricas** (`Age`, `TB`, `DB`, `Alkphos`, `Sgpt`, `Sgot`, `TP`, `ALB`, `A/G Ratio` — mezcla de `int64` y `float64`), **1 es categórica** (`Gender`, texto Male/Female) y **1 es el target** (`Selector`, `int64`, codificado 1/2). No hay columnas de identificador ni de fecha.

Nota sobre los tipos: `Alkphos`, `Sgpt` y `Sgot` son enteros porque los equipos de laboratorio que las miden (fosfatasa alcalina, ALT, AST) reportan unidades enteras (IU/L); `TB`, `DB`, `TP`, `ALB` y `A/G Ratio` son decimales por la precisión de la medición bioquímica.

### Qué mide cada variable — los tres trabajos del hígado

Describir la estructura del dataset no es solo enumerar tipos de datos: sin saber **qué mide cada columna**, los estadísticos de T3–T5 no se pueden interpretar. Las 9 variables numéricas no son una lista arbitraria — miden **tres funciones distintas** del hígado, y la dirección de la alarma **no es la misma en las tres**.

#### Eje 1 — Daño celular · ¿se están rompiendo las células?

Las células del hígado guardan enzimas **en su interior**. Si la célula se rompe, esas enzimas se derraman a la sangre. Encontrarlas elevadas en sangre es como encontrar aceite de motor en el suelo del garaje: debería estar dentro, no fuera.

| Columna | Nombre clínico | Qué mide | Unidad |
|---|---|---|---|
| `Sgpt` | **ALT** (alanina aminotransferasa) | Enzima intracelular del hepatocito. La **más específica** del hígado. | U/L |
| `Sgot` | **AST** (aspartato aminotransferasa) | Misma idea, pero AST **también existe en corazón y músculo** → menos específica. | U/L |

→ **Alto = malo.**

#### Eje 2 — Excreción biliar · ¿está drenando la bilis?

El hígado elimina desechos por la vía biliar. Si esa vía se obstruye, los desechos se acumulan en sangre.

| Columna | Nombre clínico | Qué mide | Unidad |
|---|---|---|---|
| `TB` | Bilirrubina **total** | Desecho de glóbulos rojos viejos. Si se acumula → ictericia (piel amarilla). | mg/dL |
| `DB` | Bilirrubina **directa** | La fracción que el hígado **ya procesó**. Está **contenida dentro** de `TB`. | mg/dL |
| `Alkphos` | **ALP** (fosfatasa alcalina) | Enzima de los conductos biliares. **También del hueso** — ver T2. | U/L |

→ **Alto = malo.**

#### Eje 3 — Función sintética · ¿sigue fabricando?

El hígado es una fábrica de proteínas. Si la fábrica falla, la producción cae. Como estas proteínas tienen vida media larga, este eje refleja daño **crónico**, no agudo.

| Columna | Nombre clínico | Qué mide | Unidad |
|---|---|---|---|
| `TP` | Proteína total | Toda la producción proteica en sangre. | g/dL |
| `ALB` | Albúmina | La principal proteína fabricada por el hígado. | g/dL |
| `A/G Ratio` | Cociente albúmina/globulina | **Derivada:** `ALB / (TP − ALB)`. | — |

→ **Bajo = malo.** ⚠️ Dirección **inversa** a los otros dos ejes.

#### Por qué esta agrupación importa para el resto del notebook

1. **T5 (correlación):** los pares que van a aparecer correlacionados **no son hallazgos empíricos, son estructurales** — `TB↔DB` porque una contiene a la otra, y `TP↔ALB↔A/G Ratio` por dependencia algebraica. Interpretarlos como "descubrimientos" sería un error.
2. **T4 (asimetría):** la cola derecha de las enzimas **no es ruido**, son los pacientes con daño agudo.
3. **`Selector` no es verdad biológica**, es el juicio de un especialista (*proxy label*), y el dataset **no registra la etiología** (ambigüedad A1 del PRD): no sabemos si el daño viene de alcohol, hepatitis viral u otra causa.

> Detalle completo, con unidades, rangos de referencia y fuentes primarias, en **`docs/data_dictionary.md`**.

## T2 — Problemas de calidad

¿Qué problemas de calidad existen en el dataset, como datos faltantes o errores?

In [3]:
print("--- Q2: valores faltantes por columna ---")
print(df.isnull().sum()[df.isnull().sum() > 0])

print("\n--- Q3: duplicados exactos ---")
print("Filas duplicadas:", df.duplicated().sum())
print("Filas unicas:", df.shape[0] - df.duplicated().sum())

print("\n--- Q4: desbalance de clase (Selector) ---")
print(df["Selector"].value_counts())
print((df["Selector"].value_counts(normalize=True) * 100).round(1))

print("\n--- Q5: desbalance de sexo (Gender) ---")
print(df["Gender"].value_counts())
print((df["Gender"].value_counts(normalize=True) * 100).round(1))

print("\n--- Q6: codificacion del target ---")
print("Valores unicos de Selector:", sorted(df["Selector"].unique()))

print("\n--- Q7: rango de edad ---")
print(df["Age"].describe())
print("Pacientes con Age == 90:", (df["Age"] == 90).sum())

--- Q2: valores faltantes por columna ---
A/G Ratio    4
dtype: int64

--- Q3: duplicados exactos ---
Filas duplicadas: 13
Filas unicas: 570

--- Q4: desbalance de clase (Selector) ---
Selector
1    416
2    167
Name: count, dtype: int64
Selector
1    71.4
2    28.6
Name: proportion, dtype: float64

--- Q5: desbalance de sexo (Gender) ---
Gender
Male      441
Female    142
Name: count, dtype: int64
Gender
Male      75.6
Female    24.4
Name: proportion, dtype: float64

--- Q6: codificacion del target ---
Valores unicos de Selector: [np.int64(1), np.int64(2)]

--- Q7: rango de edad ---
count    583.000000
mean      44.746141
std       16.189833
min        4.000000
25%       33.000000
50%       45.000000
75%       58.000000
max       90.000000
Name: Age, dtype: float64
Pacientes con Age == 90: 1


**Interpretación — problemas de calidad detectados por inspección univariada:**

1. **Valores faltantes:** solo `A/G Ratio` tiene nulos, **4 de 583** (0.7%). Es la única columna derivada algebraicamente (`ALB / (TP − ALB)`), lo que sugiere que los 4 faltantes ocurrieron en el cálculo original, no en la medición de `TP`/`ALB`. Justifica la Tarea 6 (imputación).

   > ⚠️ **La ficha oficial de UCI declara "Missing values: No".** La documentación contradice al archivo real. Es un recordatorio operativo: se audita el dato, no la metadata. Quien confiara en la ficha se saltaría la Tarea 6 completa.

2. **Filas duplicadas exactas:** **13 duplicados** (570 filas únicas de 583). El enunciado pregunta explícitamente por "errores" en T2, así que esto se trata como un hallazgo obligatorio, no un extra — la decisión sobre qué hacer con ellos (¿pacientes distintos con analítica idéntica, o error de captura?) se documenta en la Fase 3 (T2/F3-R13), no aquí, para no alterar todavía el dataset de referencia de este notebook.
3. **Desbalance de clase:** **416 pacientes clase 1 (enfermo) vs. 167 clase 2 (sano)**, ≈71.4% / 28.6%. La *accuracy* será una métrica engañosa en el modelado futuro (un clasificador que dijera "todos enfermos" acertaría 71% sin aprender nada).
4. **Desbalance de sexo:** **441 hombres vs. 142 mujeres**, ≈75.6% / 24.4%. Esta es la raíz técnica del sesgo documentado por Straw & Wu (2022) — con tan pocas mujeres en la muestra, cualquier modelo tiene menos señal para aprender patrones específicos de ese subgrupo.
5. **Codificación contraintuitiva del target:** `Selector` toma valores **{1, 2}**, donde **1 = enfermo** y 2 = sano — no es el 0/1 estándar y el orden no es el intuitivo. Deberá recodificarse antes de cualquier modelado.
6. **Rango de edad:** de **4 a 90 años**, con un único paciente en el extremo superior. **El valor 90 no es una edad real:** la documentación oficial de UCI establece que *"any patient whose age exceeded 89 is listed as being of age '90'"*, es decir, `Age` está **censurada por la derecha** (*top-coding*) por anonimización. Se confirma en la sección de coherencia, abajo.

   > 🔁 Esto **corrige** el estado anterior de este notebook, que reportaba la censura como "verificada, no concluyente" por falta de evidencia documental. La evidencia existía en la ficha de UCI; se localizó en el Loop C.

7. **Sin errores de formato:** no se encontraron valores no numéricos en columnas numéricas, ni categorías inesperadas en `Gender` (solo Male/Female), ni columnas con tipo de dato inconsistente.

**Pero la inspección univariada no basta.** Todos los chequeos anteriores miran **una columna a la vez**: valores faltantes, rangos, categorías. Hay errores que **solo aparecen al cruzar columnas** — un valor puede ser perfectamente plausible por sí solo e imposible en combinación con otro. La siguiente sección hace ese cruce.

### T2 (cont.) — Coherencia bioquímica y calidad de la captura

Tres chequeos que la inspección columna a columna **no puede detectar**:

1. **Restricciones de contención.** Ciertas parejas de variables se relacionan por definición, no por correlación: `DB` es una *fracción* de `TB`, y `ALB` está *contenida* en `TP`. Violarlas es imposible fisiológicamente — solo puede ser error de medición o de captura. Es lo que el marco de calidad de datos de **Kahn et al. (2016)**, adoptado por OHDSI, clasifica como violación de ***plausibilidad atemporal***.
2. ***Age heaping*.** Si las edades se estimaron o se preguntaron de memoria en vez de leerse de un documento, se acumulan artificialmente en los números redondos. Se cuantifica con el **índice de Whipple** (demografía estándar, escala de Naciones Unidas).
3. **Censura de edad (*top-coding*).** Confirmar documentalmente si el valor máximo es una edad real o un tope administrativo.

> Fuentes: `docs/fuentes/Consulta_4.md` (Kahn/OHDSI, Whipple) y la ficha oficial de UCI.

In [4]:
print("=== Q8: restricciones bioquimicas duras ===")
violations = flag_biochemical_violations(df)
print(f"DB > TB (imposible: la directa es fraccion de la total): {int(violations['db_gt_tb'].sum())} filas")
print(f"ALB > TP (imposible: la albumina esta contenida en la proteina total): {int(violations['alb_gt_tp'].sum())} filas")

print("\nFilas que violan DB <= TB:")
display(df.loc[violations["db_gt_tb"]])

print("\n=== Q9: age heaping (indice de Whipple) ===")
iw = whipple_index(df["Age"])
print(f"Indice de Whipple = {iw:.1f}  ->  '{whipple_label(iw)}' (escala ONU)")
print("Referencia: 100 = sin redondeo | <105 muy exacta | 110-125 aproximada | 125-175 tosca")

edades_redondas = df["Age"].between(23, 62) & (df["Age"] % 5 == 0)
en_rango = df["Age"].between(23, 62)
print(f"\nPersonas en edades multiplo de 5 dentro de 23-62: {int(edades_redondas.sum())}")
print(f"Esperadas si no hubiera redondeo (1 de cada 5): {en_rango.sum() / 5:.1f}")
print(f"Exceso: {100 * (iw - 100) / 100:.0f}% mas de las que deberia haber")

print("\n=== Q7: top-coding de la edad (confirmado por documentacion UCI) ===")
print(f"Pacientes con Age == {AGE_TOP_CODE}: {int((df['Age'] == AGE_TOP_CODE).sum())}")
print(f"Pacientes con Age entre 86 y 89: {int(df['Age'].between(86, 89).sum())}")
print("Doc. UCI: 'Any patient whose age exceeded 89 is listed as being of age 90.'")
print("-> Age == 90 significa '>=90', no '90 anos'. Variable censurada por la derecha.")

=== Q8: restricciones bioquimicas duras ===
DB > TB (imposible: la directa es fraccion de la total): 3 filas
ALB > TP (imposible: la albumina esta contenida en la proteina total): 0 filas

Filas que violan DB <= TB:


,Age,Gender,TB,DB,Alkphos,Sgpt,Sgot,TP,ALB,A/G Ratio,Selector
246,55,Male,1.8,9.0,272,22,79,6.1,2.7,0.7,1
261,33,Male,1.5,7.0,505,205,140,7.5,3.9,1.0,1
279,48,Female,1.0,1.4,144,18,14,8.3,4.2,1.0,1



=== Q9: age heaping (indice de Whipple) ===
Indice de Whipple = 163.3  ->  'Tosca' (escala ONU)
Referencia: 100 = sin redondeo | <105 muy exacta | 110-125 aproximada | 125-175 tosca

Personas en edades multiplo de 5 dentro de 23-62: 144
Esperadas si no hubiera redondeo (1 de cada 5): 88.2
Exceso: 63% mas de las que deberia haber

=== Q7: top-coding de la edad (confirmado por documentacion UCI) ===
Pacientes con Age == 90: 1
Pacientes con Age entre 86 y 89: 0
Doc. UCI: 'Any patient whose age exceeded 89 is listed as being of age 90.'
-> Age == 90 significa '>=90', no '90 anos'. Variable censurada por la derecha.


**Interpretación — dos problemas de calidad nuevos (Q8 y Q9) y uno resuelto (Q7).**

#### Q8 — Tres filas bioquímicamente imposibles

| Edad | Sexo | `TB` | `DB` | Lectura |
|---|---|---|---|---|
| 55 | Male | 1.8 | **9.0** | Probable transposición de dígitos |
| 33 | Male | 1.5 | **7.0** | Probable transposición de dígitos |
| 48 | Female | 1.0 | **1.4** | Error de redondeo o de captura |

La bilirrubina directa es una **fracción** de la total: `DB ≤ TB` se cumple por definición. Estas tres filas la violan.

**Por qué la inspección univariada no las vio:** un `DB` de 9.0 es un valor perfectamente plausible *por sí solo* — está dentro del rango observado de la columna (0.1–19.7). El error solo existe **en relación con `TB`**.

**Qué las causa.** La literatura de bioquímica clínica documenta que `DB > TB` puede salir del instrumento por **interferencia analítica** — paraproteínas en gammapatías monoclonales (mieloma múltiple), hemólisis, lipemia, o problemas de calibración del ensayo de bilirrubina directa. **Ningún mecanismo fisiológico real produce este resultado**: siempre es artefacto.

**Decisión de tratamiento (se ejecuta en Fase 3, no aquí):** marcar **ambas** columnas (`TB` y `DB`) como faltantes en esas 3 filas, **sin eliminar la fila**. Dos razones: (a) no hay evidencia de cuál de los dos valores está mal, así que "corregir" sería inventar; (b) el resto de la analítica de esos 3 pacientes es válida y eliminar filas es el peor método de la jerarquía del §9.4 del PRD. Es la práctica que recomiendan Kahn/OHDSI para violaciones de plausibilidad no explicables.

`ALB > TP`: **0 violaciones** — la restricción de contención del eje de síntesis se cumple en las 583 filas.

#### Q9 — Las edades no se leyeron de un documento

**Índice de Whipple = 163.3 → categoría "Tosca"** en la escala de Naciones Unidas (banda 125–175).

El índice compara cuánta gente cae en edades terminadas en 0 o 5 (dentro del rango canónico 23–62) contra cuánta debería caer si no hubiera redondeo. Un valor de 100 significa ausencia total de preferencia de dígito; 500 es el máximo teórico. **163.3 significa un 63% más de personas en edades redondas de las que debería haber.**

**Qué implica:** las edades se **estimaron o se reportaron de memoria**, no se verificaron contra un documento de identidad. Consecuencias prácticas:

- `Age` tiene un **error de medición estimado de ±2–3 años**. No es una variable de precisión.
- **No usar franjas etarias estrechas** (tipo "40–44"): el ruido de redondeo se comería la señal. Las bandas de `AGE_BANDS` son deliberadamente amplias.
- Señal de alarma general: si el campo más fácil de capturar se recogió con descuido, conviene no asumir precisión en el resto.

#### Q7 — Resuelto: `Age = 90` no es una edad

Hay **1 solo paciente** con `Age = 90` y **ninguno entre 86 y 89**. La documentación oficial de UCI lo explica: *"any patient whose age exceeded 89 is listed as being of age '90'"*.

`Age = 90` significa **"≥ 90"**, no "90 años". La variable está **censurada por la derecha** (*top-coding*) por anonimización. El rango observado 4–90 **no es un rango real de edades**: su extremo superior es un artefacto administrativo.

Esto **cierra una pregunta que estaba abierta**: la versión anterior de este notebook reportaba la censura como "verificada, no concluyente" porque no se había localizado evidencia documental.

#### Sobre el paciente de 4 años — no es un error

El extremo **inferior** sí es real. La paciente de 7 años del dataset tiene `TB`=27.2, `DB`=11.8, `Alkphos`=1420, `Sgpt`=790, `Sgot`=1050: un cuadro de hepatitis aguda internamente coherente. La enfermedad hepática pediátrica existe (atresia biliar, Wilson, hepatitis A, metabolopatías).

**Pero los 25 menores de 18 años sí abren un problema de validez metodológica**, distinto de un problema de calidad — se cuantifica más abajo, en la sección de `Alkphos` por edad.

---

**Recuento actualizado: 9 problemas de calidad** (Q1–Q7 de la inspección univariada, más Q8 y Q9 de la coherencia multivariada). Tabla consolidada en `docs/data_dictionary.md` §4.

## T3 — Estadística descriptiva de `TB` y `DB`

¿Cuáles son la media, la desviación estándar, la varianza, el rango y el rango intercuartílico de `TB` y `DB`? ¿Qué implicaciones tienen para la variabilidad de los datos?

> Se calcula sobre los datos **originales, sin imputar** — ninguna de las dos columnas tiene faltantes, así que no hay diferencia con la versión de la Fase 3, pero se deja explícito por consistencia con la regla del PRD de no mezclar estadísticos con datos ya tratados.

In [5]:
rows = []
for col in ["TB", "DB"]:
    s = df[col]
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    rows.append({
        "variable": col,
        "media": s.mean(),
        "mediana": s.median(),
        "std": s.std(),
        "varianza": s.var(),
        "min": s.min(),
        "max": s.max(),
        "rango": s.max() - s.min(),
        "IQR": q3 - q1,
    })
t3_summary = pd.DataFrame(rows).set_index("variable").round(3)
t3_summary

,media,mediana,std,varianza,min,max,rango,IQR
variable,,,,,,,,
TB,3.299,1.0,6.210,38.558,0.4,75.0,74.6,1.8
DB,1.486,0.3,2.808,7.888,0.1,19.7,19.6,1.1


**Interpretación.**

| Variable | Media | Mediana | Std | Varianza | Rango | IQR |
|---|---|---|---|---|---|---|
| `TB` | 3.30 | 1.00 | 6.21 | 38.56 | 74.60 (0.4–75.0) | 1.80 |
| `DB` | 1.49 | 0.30 | 2.81 | 7.89 | 19.60 (0.1–19.7) | 1.10 |

**Media vs. mediana — evidencia de asimetría fuerte:** en ambas variables la media es **muy superior** a la mediana (`TB`: 3.30 vs. 1.00; `DB`: 1.49 vs. 0.30) — la media casi triplica/cuadruplica a la mediana. Esto es la firma numérica de una distribución con **cola derecha larga**: unos pocos pacientes con bilirrubina muy alta arrastran la media hacia arriba, mientras que la mayoría de los pacientes tiene valores bajos, cercanos a la mediana. Se confirma con *skewness*/*kurtosis* en T4.

**Rango total vs. IQR — cuánto "estiran" los extremos:** el rango de `TB` (74.6) es **~41 veces** su IQR (1.8); el de `DB` (19.6) es **~18 veces** su IQR (1.1). El IQR —robusto a extremos— describe la dispersión del 50% central de los pacientes, y es mucho más pequeño que el rango total. La brecha enorme entre ambos indica que un número reducido de pacientes está "estirando" la distribución muy por encima de donde vive la mayoría de los datos.

---

**Lectura clínica — qué significan estos números en un paciente real.**

`TB` y `DB` pertenecen al **eje de excreción biliar**: miden desecho de glóbulos rojos viejos que el hígado debería estar eliminando. **Alto = malo.** Cuando la bilirrubina se acumula lo suficiente, aparece la ictericia (piel y ojos amarillos).

Contra el rango de referencia adulto (`TB`: 0.3–1.2 mg/dL · `DB`: 0.0–0.3 mg/dL), estos estadísticos dicen algo que la estadística sola no revela:

| | Mediana | Techo normal | Lectura |
|---|---|---|---|
| `TB` | **1.00** | 1.2 | El paciente mediano del dataset tiene bilirrubina total **dentro de lo normal** |
| `DB` | **0.30** | 0.3 | El paciente mediano está **justo en el límite** |

**Esto es un hallazgo, no un detalle.** El dataset se llama "Indian Liver Patient" y el 71% de sus filas están etiquetadas como hepáticas — pero **la mitad de la muestra tiene bilirrubina normal**. La ictericia franca es un signo **tardío**: aparece cuando el daño ya está avanzado. La mayoría de estos pacientes están en fases donde el hígado todavía procesa la bilirrubina adecuadamente, y su enfermedad, si existe, se manifiesta en **otros ejes** (daño celular vía `Sgpt`/`Sgot`, o función sintética vía `ALB`).

**Consecuencia directa para la variabilidad:** la enorme varianza de `TB` (38.56) **no está distribuida entre todos los pacientes** — está concentrada en la minoría de la cola. Estadísticamente, esto significa que media y desviación estándar son **malos resúmenes** de esta variable: describen a un paciente promedio que casi no existe. Mediana e IQR describen mejor a la población real.

**Y los extremos no son ruido a eliminar:** un `TB` de 75.0 mg/dL —62 veces el techo normal— es un paciente con obstrucción biliar severa o fallo hepático. Es exactamente **el caso que una herramienta de cribado debe detectar**. Eliminarlo como "outlier" subiría la tasa de falsos negativos justo donde más daño hace (se retoma en T8, Fase 3).

## T4 — Forma y simetría de `TB`, `DB`, `Alkphos`, `Sgpt`

¿Qué forma y simetría tienen las distribuciones? Histogramas y análisis de los patrones observados.

In [6]:
t4_cols = ["TB", "DB", "Alkphos", "Sgpt"]

fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for ax, col in zip(axes.flat, t4_cols):
    sns.histplot(df[col], bins=30, kde=True, ax=ax, color="#4C72B0")
    ax.set_title(col)
    ax.set_xlabel(col)
fig.suptitle("T4 — Distribuciones de biomarcadores (dataset completo, n=583)")
fig.tight_layout()
save_figure(fig, "t4_hist_bilirubin_enzymes.png")
plt.show()

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_44988\2098913191.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
skew_kurt = pd.DataFrame({
    "skewness": df[NUMERIC_COLS].skew(),
    "kurtosis_fisher": df[NUMERIC_COLS].kurtosis(),
}).round(3)
skew_kurt

,skewness,kurtosis_fisher
Age,-0.029,-0.560
TB,4.907,37.164
DB,3.212,11.353
Alkphos,3.765,17.753
Sgpt,6.549,50.579
Sgot,10.546,150.920
TP,-0.286,0.233
ALB,-0.044,-0.388
A/G Ratio,0.992,3.282


**Interpretación.** Los cuatro histogramas muestran el mismo patrón: una barra alta cerca de cero y una cola larga hacia la derecha, con muy pocos casos aislados en valores extremos. La *skewness* lo confirma numéricamente:

- `TB`: skew = **4.91**, kurtosis = **37.2**
- `DB`: skew = **3.21**, kurtosis = **11.4**
- `Alkphos`: skew = **3.77**, kurtosis = **17.8**
- `Sgpt`: skew = **6.55**, kurtosis = **50.6**

Todas tienen **skewness fuertemente positivo** (>0, cola derecha) y **kurtosis muy por encima de 0** (colas mucho más pesadas que una normal — la referencia de "sin exceso" es kurtosis de Fisher = 0). `Sgot` (no graficada aquí, se retoma en T8) es la más extrema de las nueve variables numéricas: skew = 10.55, kurtosis = 150.9.

---

**Lectura clínica — la asimetría sigue el eje funcional, y eso no es casualidad.**

Si se ordenan las nueve variables por *skewness*, el resultado **reproduce exactamente la agrupación por ejes** presentada en T1:

| Eje funcional | Variables | Skewness | Forma |
|---|---|---|---|
| **Daño celular** | `Sgot` (10.55), `Sgpt` (6.55) | Extrema | Cola derecha larguísima |
| **Excreción biliar** | `TB` (4.91), `Alkphos` (3.77), `DB` (3.21) | Fuerte | Cola derecha marcada |
| **Función sintética** | `TP` (−0.29), `ALB` (−0.04), `A/G Ratio` (0.99) | ≈ Simétrica | Casi normal |
| *(Demográfica)* | `Age` (−0.03) | Simétrica | — |

**Por qué ocurre esto — la explicación es fisiológica, no estadística:**

Los dos primeros ejes miden **fugas y acumulaciones**. Cuando una célula hepática se rompe, vierte su contenido enzimático a la sangre de golpe: las enzimas pueden multiplicarse por 10, 50 o 100 respecto a su nivel basal en cuestión de días. No hay techo biológico cercano — de ahí `Sgot` con máximo 4929 U/L frente a una mediana de 42. **Esa cola son los pacientes en crisis aguda.**

El tercer eje mide **capacidad de producción**, y funciona al revés. La albúmina no puede "dispararse": el hígado tiene un techo de fabricación, y cuando falla, la producción **baja lentamente** (semanas o meses, porque la albúmina ya circulante tiene vida media larga). Una variable acotada por arriba y de cambio lento produce una distribución **simétrica y estrecha** — exactamente lo que muestran `TP` (skew −0.29) y `ALB` (skew −0.04).

**La consecuencia práctica es asimétrica, y hay que decirla:**

- En los ejes 1 y 2, **la cola derecha es la señal**: los valores extremos son los pacientes graves, justo los que un cribado debe detectar.
- En el eje 3, la señal está en el **extremo izquierdo** (albúmina baja = fábrica fallando), y como la distribución es estrecha, **una caída pequeña de `ALB` puede ser clínicamente más grave que un pico grande de `Sgpt`**.

Un tratamiento de outliers que aplique la misma regla a las nueve variables ignora esta diferencia. Se retoma en T8 (Fase 3).

**Implicación técnica para el escalado (T7, Fase 3):** con estos niveles de asimetría, MinMax es especialmente peligroso — un solo valor extremo infla el denominador `max − min` y comprime al resto contra el cero. Con `Sgot` (max 4929, mediana 42), la mediana escalada quedaría en ~0.007. Se cuantifica en F3-R9.

## T5 — Matriz de correlación

¿Qué relaciones existen entre las variables? Matriz de correlación y correlaciones significativas que puedan influir en la selección de variables.

> Se calculan **Pearson** (relación lineal) y **Spearman** (relación monótona, robusta a la fuerte asimetría de T4) — con distribuciones tan sesgadas, Spearman es la referencia más confiable.

In [8]:
corr_pearson = df[NUMERIC_COLS].corr(method="pearson")
corr_spearman = df[NUMERIC_COLS].corr(method="spearman")

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
sns.heatmap(corr_pearson, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1, vmax=1, ax=axes[0])
axes[0].set_title("Pearson")
sns.heatmap(corr_spearman, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1, vmax=1, ax=axes[1])
axes[1].set_title("Spearman")
fig.suptitle("T5 — Matriz de correlación (variables numéricas, n=583)")
fig.tight_layout()
save_figure(fig, "t5_heatmap_correlacion.png")
plt.show()

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_44988\1732473117.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
pairs = []
for i in range(len(NUMERIC_COLS)):
    for j in range(i + 1, len(NUMERIC_COLS)):
        r_p = corr_pearson.iloc[i, j]
        if abs(r_p) > CORR_THRESHOLD:
            pairs.append({
                "var_1": NUMERIC_COLS[i],
                "var_2": NUMERIC_COLS[j],
                "pearson": round(r_p, 3),
                "spearman": round(corr_spearman.iloc[i, j], 3),
            })
pares_altos = pd.DataFrame(pairs)
pares_altos

,var_1,var_2,pearson,spearman
0,TB,DB,0.875,0.959
1,Sgpt,Sgot,0.792,0.774
2,TP,ALB,0.784,0.779


**Interpretación.** Tres pares superan el umbral `CORR_THRESHOLD = 0.7`:

| Grupo funcional | Par | Pearson | Spearman |
|---|---|---|---|
| Excreción | `TB` ↔ `DB` | **0.875** | **0.959** |
| Daño celular | `Sgpt` ↔ `Sgot` | **0.792** | **0.774** |
| Síntesis | `TP` ↔ `ALB` | **0.784** | **0.779** |

(Referencia adicional, justo debajo del umbral: `ALB` ↔ `A/G Ratio`, Pearson 0.690 / Spearman 0.754.)

---

**Lectura clínica — lo primero es que estas correlaciones NO son hallazgos.**

Este es el punto más importante de T5, y es fácil de errar: **los tres pares que superan el umbral son exactamente los tres ejes funcionales definidos en T1**. Correlacionan porque **miden lo mismo por construcción**, no porque el análisis haya descubierto una relación en los datos.

Hay que distinguir tres tipos de correlación, porque **cada uno se trata distinto**:

| Tipo | Par | Por qué correlacionan | ¿Es un hallazgo? |
|---|---|---|---|
| **Contención** | `TB` ↔ `DB` | `DB` es una **fracción física** de `TB`. Medir ambas es medir el todo y una de sus partes. | ❌ No. Es definicional. |
| **Algebraica** | `TP` ↔ `ALB` ↔ `A/G Ratio` | `A/G Ratio = ALB / (TP − ALB)`. La tercera se **calcula** de las otras dos. | ❌ No. Es aritmética. |
| **Fisiológica** | `Sgpt` ↔ `Sgot` | Dos enzimas **distintas**, en órganos parcialmente distintos, que suben juntas porque **responden al mismo daño**. | ✅ **Sí.** Es el único empírico. |

Reportar `TB↔DB` = 0.96 como "descubrimos que las bilirrubinas están fuertemente asociadas" sería un error de interpretación: es como descubrir que el total de una factura correlaciona con uno de sus renglones.

**El par `Sgpt`↔`Sgot` es el único informativo — y su cociente dice más que su correlación.** Ambas son enzimas intracelulares que se derraman cuando el hepatocito se rompe, pero ALT (`Sgpt`) es casi exclusiva del hígado mientras AST (`Sgot`) también existe en corazón y músculo. Que correlacionen a 0.79 y no a 0.99 es precisamente lo interesante: **la parte donde NO coinciden lleva información sobre el tipo de daño**. Eso se explota en la sección del cociente De Ritis, más abajo.

---

**Conexión con la selección de variables — decisión por grupo:**

| Grupo | Variable a conservar | Razón |
|---|---|---|
| `TB` ↔ `DB` | **`TB`** | `DB` está *contenida* en `TB` — redundancia **estructural**. Spearman = 0.96 indica que `TB` sola retiene casi toda la señal ordinal. `TB` es además el marcador de cribado inicial más usado clínicamente. |
| `Sgpt` ↔ `Sgot` | **Ambas, con reserva** | La correlación es alta pero **no estructural**. Straw & Wu (2022) encuentran que **pesan distinto según el sexo** (ALT/AST rankean 4.º–5.º en mujeres vs 7.º–8.º en hombres) — eliminar una de entrada borraría señal relevante justo para el análisis de *fairness* que motiva este proyecto. Evaluar VIF en Fase 4 antes de descartar. |
| `TP` ↔ `ALB` (+ `A/G Ratio`) | **`ALB` y `A/G Ratio`** | Dependencia algebraica. `ALB` es el biomarcador de síntesis más directo e interpretable; `A/G Ratio` ya combina `TP` y `ALB`. `TP` es la más redundante de las tres. |

**Advertencia sobre `A/G Ratio` como variable derivada.** Al ser función de `ALB` y `TP`, no aporta información nueva: aporta una **transformación** de información existente. Si se conservan las tres, un modelo lineal sufrirá multicolinealidad severa (la matriz de diseño es casi singular). Su valor está en la interpretabilidad clínica, no en la información.

**Lectura general:** correlación alta entre predictores significa **redundancia**, no causalidad. La decisión final de qué excluir debe tomarse en la Fase 4 con herramientas cuantitativas (VIF, importancia de variables) — y sabiendo que **dos de los tres pares son redundancia por definición, no por evidencia.**

## Valor agregado — Análisis estratificado por `Gender`

El enunciado no lo exige, pero es el eje diferenciador de este proyecto (§0.2 del PRD). Se hace **después** del análisis agregado, nunca en su reemplazo.

In [10]:
print("--- Medias por sexo (variables numéricas) ---")
display(df.groupby("Gender")[NUMERIC_COLS].mean().round(3))

print("\n--- Tabla de contingencia Gender x Selector ---")
display(pd.crosstab(df["Gender"], df["Selector"]))

print("\n--- Tasa de diagnóstico positivo (Selector==1) por sexo ---")
tasa_positivo = df.groupby("Gender")["Selector"].apply(lambda s: (s == 1).mean() * 100).round(2)
display(tasa_positivo)

--- Medias por sexo (variables numéricas) ---


,Age,TB,DB,Alkphos,Sgpt,Sgot,TP,ALB,A/G Ratio
Gender,,,,,,,,,
Female,43.134,2.323,0.989,302.338,54.239,69.042,6.654,3.273,0.949
Male,45.265,3.613,1.646,286.789,89.238,123.070,6.428,3.100,0.946



--- Tabla de contingencia Gender x Selector ---


Selector,1,2
Gender,,
Female,92,50
Male,324,117



--- Tasa de diagnóstico positivo (Selector==1) por sexo ---


Gender
Female    64.79
Male      73.47
Name: Selector, dtype: float64

In [11]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, col in zip(axes, ["TB", "Alkphos"]):
    sns.histplot(data=df, x=col, hue="Gender", bins=30, kde=True, ax=ax, palette=["#4C72B0", "#DD8452"], element="step")
    ax.set_title(f"{col} por sexo")
fig.suptitle("Distribuciones estratificadas por Gender")
fig.tight_layout()
save_figure(fig, "eda_extra_hist_por_sexo.png")
plt.show()

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_44988\4068109723.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Interpretación.** Dos hallazgos relevantes para la pregunta de investigación 2 (¿hay diferencias de biomarcadores por sexo?):

1. **Los hombres tienen enzimas y bilirrubinas más altas en promedio:** `TB` (3.61 vs. 2.32), `DB` (1.65 vs. 0.99), `Sgpt` (89.2 vs. 54.2), `Sgot` (123.1 vs. 69.0) — todas más altas en hombres que en mujeres. Es consistente con que la muestra masculina tiene proporcionalmente más casos graves (ver tasa de diagnóstico abajo).
2. **`Alkphos` (ALP) rompe el patrón:** es la **única** enzima con media más alta en mujeres (302.3) que en hombres (286.8), pese a que las mujeres tienen valores más bajos en todo lo demás. Esto coincide con lo que reporta Straw & Wu (2022): al corregir la subrepresentación femenina, **ALP y sexo se vuelven las variables más importantes** del modelo — este dataset ya muestra la semilla de ese patrón a nivel descriptivo.

**Tasa de diagnóstico positivo por sexo — hallazgo que conecta con el *label bias* (Sección 2, Fase 1):** el 73.47% de los hombres tiene `Selector=1` (enfermo) frente al **64.79% de las mujeres** (tabla de contingencia: 324/117 hombres, 92/50 mujeres). A partir de biomarcadores más bajos en promedio, es razonable que la tasa de diagnóstico también sea menor en mujeres — pero **no podemos distinguir, solo con este dataset, si esa diferencia refleja una población realmente más sana o un patrón de sub-diagnóstico** (el segundo es justamente lo que reporta la literatura citada en la Fase 1 sobre menor sospecha clínica en mujeres). Se deja como pregunta abierta explícita para la auditoría de *fairness* de la Fase 5 (fuera de este PRD), no se puede responder con EDA.

> 🔁 **Nota de iteración (Loop A):** este hallazgo (diferencia de ~9 puntos porcentuales en tasa de diagnóstico por sexo, más el patrón atípico de `Alkphos`) es justo el tipo de resultado que la Fase 1 anticipaba que podía reforzar la pregunta de investigación 3. Se registra en `docs/CHANGELOG_iteraciones.md`.

## Valor agregado — Verificación de la fórmula `A/G Ratio = ALB / (TP − ALB)`

Prepara la decisión de imputación determinista de la Fase 3 (F3-R5 del PRD): si la fórmula reconstruye `A/G Ratio` con error bajo, es una alternativa superior a imputar por media.

In [12]:
print("Filas con TP == ALB (riesgo de división por cero):", (df["TP"] == df["ALB"]).sum())

comp = df.dropna(subset=["A/G Ratio", "TP", "ALB"]).copy()
comp["AG_calc"] = comp["ALB"] / (comp["TP"] - comp["ALB"])
comp["error_abs"] = (comp["AG_calc"] - comp["A/G Ratio"]).abs()

print(f"\nFilas comparables (sin nulos): {len(comp)}")
print("\nError absoluto de reconstrucción:")
print(comp["error_abs"].describe().round(4))
print("\nFilas con error > 0.05:", (comp["error_abs"] > 0.05).sum(), f"({(comp['error_abs'] > 0.05).mean()*100:.1f}%)")

Filas con TP == ALB (riesgo de división por cero): 0

Filas comparables (sin nulos): 579

Error absoluto de reconstrucción:
count    579.0000
mean       0.0514
std        0.1664
min        0.0000
25%        0.0067
50%        0.0310
75%        0.0615
max        2.3000
Name: error_abs, dtype: float64

Filas con error > 0.05: 191 (33.0%)


**Interpretación — hallazgo más matizado de lo esperado.** No hay riesgo de división por cero (0 filas con `TP == ALB`). Sobre las 579 filas comparables, la reconstrucción algebraica **no es perfectamente exacta**: error absoluto medio = **0.051**, mediana = **0.031**, pero con una cola de casos peores (máximo 2.30) y **191 filas (33.0%) con error > 0.05**.

Esto no invalida la fórmula — la relación `A/G Ratio = ALB / (TP − ALB)` es correcta por definición bioquímica (globulina = proteína total − albúmina) — pero indica que **los valores originales del dataset están redondeados** antes de guardarse (`TP` y `ALB` a un decimal, `A/G Ratio` a dos), y dividir dos cantidades ya redondeadas amplifica el error relativo, especialmente cuando `TP − ALB` (la globulina) es pequeña. Es un matiz importante para la Fase 3: la imputación determinista sigue siendo **conceptualmente superior** a la media (no depende de la distribución de otros pacientes), pero no es "exacta" en la práctica sobre este dataset concreto — se reportará así, con honestidad, en vez de asumir un error cercano a cero.

## Valor agregado — `Alkphos` depende de la edad, y eso confunde la detección de outliers

**El problema, en una frase:** la fosfatasa alcalina **no viene solo del hígado**. Los osteoblastos —las células que construyen hueso— la producen para poder mineralizar. **Un niño en crecimiento tiene ALP alta de forma completamente normal.**

Rangos de referencia pediátricos publicados:

| Edad | Sexo | ALP normal (U/L) |
|---|---|---|
| 1–10 años | ambos | 142 – 335 |
| 10–13 años | ambos | 129 – 417 |
| 13–15 años | niños | 116 – **468** ← pico del estirón |
| 13–15 años | niñas | 57 – 254 |
| 15–17 años | niños | 82 – 331 |
| 15–17 años | niñas | 50 – 117 |
| **>19 años (adulto)** | ambos | **40 – 129** |

*(Children's Minnesota Lab; CALIPER; Zierk et al. 2017. Las niñas bajan antes porque entran a la pubertad antes.)*

**Por qué importa aquí:** este dataset mezcla **25 menores de 18 años con 558 adultos**. La Tarea 8 (Fase 3) pide detectar valores atípicos con la regla de Tukey (1.5·IQR), que calcula los cuartiles sobre **toda** la población. Como los adultos dominan la muestra, el umbral de "atípico" se calibra con ellos — y los niños sanos con ALP fisiológicamente alta caerán fuera.

**No sería un outlier clínico: sería un artefacto de mezclar dos poblaciones.**

La norma internacional de laboratorios clínicos (**CLSI C28-A3**) y el proyecto de referencia pediátrica **CALIPER** son explícitos: se **particiona por edad y sexo primero**, y solo **después** se aplica Tukey dentro de cada partición. Hacerlo al revés se considera error de diseño.

> Fuente: `docs/fuentes/Consulta_2.md`.

In [13]:
band_labels = [f"{lo}-{hi}" if hi < 120 else f"{lo}+" for lo, hi in AGE_BANDS]


def assign_age_band(age: int) -> str:
    """Devuelve la etiqueta de franja etaria de AGE_BANDS que contiene a `age`."""
    for (lo, hi), label in zip(AGE_BANDS, band_labels):
        if lo <= age <= hi:
            return label
    return "sin banda"


df_bands = df.assign(banda_edad=df["Age"].apply(assign_age_band))
df_bands["banda_edad"] = pd.Categorical(df_bands["banda_edad"], categories=band_labels, ordered=True)

print("--- Alkphos por franja de edad ---")
alp_por_edad = df_bands.groupby("banda_edad", observed=True)["Alkphos"].agg(
    n="size", media="mean", mediana="median", q1=lambda s: s.quantile(0.25),
    q3=lambda s: s.quantile(0.75), maximo="max"
).round(1)
display(alp_por_edad)


def tukey_upper_bound(s: pd.Series) -> float:
    """Limite superior de la regla de Tukey: Q3 + IQR_MULTIPLIER * IQR."""
    q1, q3 = s.quantile([0.25, 0.75])
    return q3 + IQR_MULTIPLIER * (q3 - q1)


# Cuantos MENORES serian marcados como atipicos por un Tukey calculado sobre
# TODA la poblacion (adultos + menores), que es lo que pide la rubrica en T8.
limite_global = tukey_upper_bound(df["Alkphos"])
print(f"\nTukey GLOBAL sobre Alkphos: limite superior = {limite_global:.1f} U/L")

menores = df_bands[df_bands["Age"] < AGE_ADULT_MIN]
n_atipicos_global = int((menores["Alkphos"] > limite_global).sum())
print(f"Menores de {AGE_ADULT_MIN} marcados como atipicos por el Tukey global: "
      f"{n_atipicos_global} de {len(menores)} ({100 * n_atipicos_global / len(menores):.1f}%)")

# Tukey estratificado: calculado SOLO dentro de la franja pediatrica
limite_ped = tukey_upper_bound(menores["Alkphos"])
n_atipicos_ped = int((menores["Alkphos"] > limite_ped).sum())
print(f"\nTukey ESTRATIFICADO (solo menores): limite superior = {limite_ped:.1f} U/L")
print(f"Menores atipicos con el umbral de su propia poblacion: "
      f"{n_atipicos_ped} de {len(menores)} ({100 * n_atipicos_ped / len(menores):.1f}%)")

--- Alkphos por franja de edad ---


,n,media,mediana,q1,q3,maximo
banda_edad,,,,,,
0-17,25,385.7,320.0,268.0,401.0,1420
18-39,200,249.4,198.0,165.0,279.2,2110
40-59,225,296.4,210.0,178.0,292.0,1896
60+,133,324.8,215.0,178.0,356.0,1750



Tukey GLOBAL sobre Alkphos: limite superior = 481.8 U/L
Menores de 18 marcados como atipicos por el Tukey global: 4 de 25 (16.0%)

Tukey ESTRATIFICADO (solo menores): limite superior = 600.5 U/L
Menores atipicos con el umbral de su propia poblacion: 2 de 25 (8.0%)


In [14]:
ALP_VIEW_MAX = 1500  # recorte de vista; los maximos adultos llegan a 2110

fig, ax = plt.subplots(figsize=(9.5, 5.8))

sns.boxplot(
    data=df_bands, x="banda_edad", y="Alkphos", hue="banda_edad",
    palette=AGE_BAND_COLORS, legend=False, ax=ax,
    fliersize=3, linewidth=1.2, width=0.55, saturation=1,
)

# Banda de referencia ADULTA (40-129 U/L): solo interpretable para adultos.
ax.axhspan(40, 129, color=INK_MUTED, alpha=0.18, zorder=0)
ax.text(-0.46, 84, "ref. adulto 40\u2013129 U/L", va="center", ha="left",
        fontsize=8.5, color=INK_SECONDARY, style="italic")

# Limite de Tukey calculado sobre TODA la poblacion (lo que pide T8)
ax.axhline(limite_global, color=INK_PRIMARY, linestyle="--", linewidth=1.6, zorder=4)
ax.text(-0.46, limite_global + 45, f"Tukey global = {limite_global:.0f} U/L",
        fontsize=9, color=INK_PRIMARY, fontweight="bold")

# Etiqueta directa de la mediana, FUERA de la caja para que se lea siempre
for i, etiqueta in enumerate(band_labels):
    sub = df_bands.loc[df_bands["banda_edad"] == etiqueta, "Alkphos"]
    ax.text(i + 0.33, sub.median(), f"mediana\n{sub.median():.0f}", ha="left",
            va="center", fontsize=9, fontweight="bold", color=INK_SECONDARY)
    ax.text(i, -95, f"n={len(sub)}", ha="center", va="center",
            fontsize=8.5, color=INK_MUTED)

ax.set_ylim(-160, ALP_VIEW_MAX)
ax.set_xlim(-0.55, len(band_labels) - 0.25)
ax.set_xlabel("Franja de edad (a\u00f1os)", labelpad=18)
ax.set_ylabel("Alkphos \u2014 ALP (U/L)")
ax.set_title("ALP por franja de edad\nLos menores tienen ALP alta por crecimiento \u00f3seo, no por da\u00f1o hep\u00e1tico",
             fontsize=11, pad=12)
ax.set_yticks(range(0, ALP_VIEW_MAX + 1, 250))
ax.grid(axis="y", color=GRID_COLOR)
ax.grid(axis="x", visible=False)

fig.tight_layout()
save_figure(fig, "loopc_alp_por_edad.png")
plt.show()

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_44988\248418153.py:41: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Interpretación — la edad confunde la detección de outliers de ALP.**

La mediana de ALP en menores (**320 U/L**) es sustancialmente más alta que en cualquier franja adulta (198–215 U/L), y esa diferencia **es fisiológica, no patológica**: los menores de este dataset caen dentro de los rangos pediátricos publicados (129–468 U/L según edad y sexo), mientras que para un adulto el techo normal es 129 U/L.

**El mismo número significa lo contrario según la edad:**

| Persona con ALP = 320 | Su rango de referencia | Veredicto |
|---|---|---|
| Niño de 13 años | 116 – 468 | ✅ **Normal** |
| Adulto de 45 años | 40 – 129 | ❌ **Muy elevado** |

### Cuantificación del efecto — y su magnitud real

| Criterio | Límite superior | Menores marcados atípicos |
|---|---|---|
| Tukey **global** (583 filas, lo que pide T8) | 481.8 U/L | **4 de 25 (16.0%)** |
| Tukey **estratificado** (solo los 25 menores) | 600.5 U/L | **2 de 25 (8.0%)** |

El umbral global **duplica** la tasa de menores marcados como atípicos respecto al umbral calculado con su propia población — pero en términos absolutos la diferencia son **2 niños**. El efecto es real y en la dirección esperada, pero **modesto en esta muestra**, y conviene decirlo así en vez de exagerarlo.

**Por qué resulta más pequeño de lo que cabría temer:** el límite de Tukey global (481.8 U/L) es *más permisivo* que el techo del rango pediátrico de referencia (468 U/L). Ocurre porque `Alkphos` tiene una cola derecha tan larga en adultos (máximos de 1896 y 2110) que infla `Q3 + 1.5·IQR` hasta un valor alto. **Es una coincidencia favorable de esta muestra, no una garantía metodológica**: con una distribución adulta menos dispersa, el umbral global habría sido más bajo y el sesgo mayor.

**Lo que sí queda establecido:** el umbral global **no guarda ninguna relación con la fisiología de los menores**. Que no los penalice más es fortuito, no correcto — y el razonamiento se agrava en variables donde la cola adulta sea más corta.

**Esto no es un problema de calidad del dato** (los valores son correctos y los pacientes reales), **sino de validez del análisis**: se trata como homogénea una población que no lo es.

---

**Decisión para la Fase 3 (T8) — añadir, nunca sustituir:**

1. **[M] Obligatorio:** ejecutar el Tukey global sobre las 583 filas, tal como exige la rúbrica. **No se toca.**
2. **[V] Valor agregado:** repetir el conteo estratificado por franja de edad y reportar la discrepancia, citando CLSI C28-A3 y CALIPER — que exigen **particionar antes** de aplicar Tukey, no después.

Esto respeta el §0.4 del PRD ("no sustituir el requisito por la alternativa") y a la vez demuestra el criterio metodológico. La misma lógica ya se aplica a la estratificación por sexo (A5): **primero el análisis agregado que se exige, después el estratificado que aporta.**

> ⚠️ **Conexión con el eje de *fairness*:** los rangos pediátricos de ALP **también difieren por sexo** (13–15 años: niños 116–468 vs niñas 57–254, porque ellas entran antes a la pubertad). Un tratamiento de outliers ciego a edad **y** a sexo acumula dos sesgos sobre el mismo subgrupo. Se retoma en F3-R15.

## ⭐ Valor agregado — Análisis de sensibilidad: el umbral de ALT y la brecha por sexo

Esta sección conecta el hallazgo descriptivo del análisis estratificado (brecha de 8.7 pp en tasa de diagnóstico) con un **mecanismo concreto, medible y citable** por el que esa brecha podría producirse.

### Qué es un umbral de referencia y por qué es una decisión, no un hecho

Un valor de laboratorio se declara "anormal" comparándolo contra un **límite superior de normalidad (ULN)**. Ese límite se construye midiendo a un grupo de personas sanas, viendo cómo se distribuyen los valores, y cortando en el 95% central. El techo de ese 95% es el ULN.

Dos consecuencias que casi nunca se explicitan:

- **El ULN es una decisión humana**, no una ley natural. Alguien eligió dónde cortar.
- **El ULN depende de a quién se midió.** Medir donantes en Milán da un número; medir población de Andhra Pradesh da otro.

### El hallazgo de Prati (2002) y su adopción

Durante décadas los laboratorios usaron **un solo ULN para ALT: 40 U/L**, igual para hombres y mujeres.

En 2002, **Prati et al.** midieron ALT en **6.835 donantes de sangre sanos** en Milán, excluyendo hepatitis C, obesidad y consumo de alcohol. Encontraron que **las mujeres sanas tienen ALT naturalmente más bajo que los hombres sanos** — fisiología normal, no mejor salud.

| Criterio | Hombres | Mujeres | Fuente |
|---|---|---|---|
| Unisex de la época del ILPD | 40 | 40 | Práctica de laboratorio 2007–2012 |
| *Healthy ULN* | **30** | **19** | Prati et al. 2002, Ann Intern Med |
| Guía clínica actual | 29–33 | 19–25 | ACG 2017 · AASLD 2023 |

**El punto clave:** un umbral único de 40 queda **mucho más cerca del techo masculino (30) que del femenino (19)**. Sin proponérselo, está calibrado para hombres.

### Qué le pasa a una persona concreta

Una mujer con **ALT = 25**:

| Umbral aplicado | Comparación | Resultado | Consecuencia |
|---|---|---|---|
| Unisex (40) | 25 < 40 | **Normal** | Se va a casa. Nadie investiga. |
| Femenino (19) | 25 > 19 | **Elevado** | Se investiga. Puede recibir diagnóstico. |

Misma mujer, misma sangre, mismo número. Dos destinos opuestos.

Un **hombre** con ALT = 25 es normal bajo **ambos** criterios (25 < 40 y 25 < 30): a él el cambio de umbral **no le afecta**.

**De ahí la hipótesis a contrastar:** *¿está la población femenina de este dataset concentrada en el rango donde la elección del umbral decide el resultado?*

> Fuente: `docs/fuentes/Consulta_1.md`. Umbrales en `src/config.py` (`ALT_ULN_UNISEX`, `ALT_ULN_BY_SEX`).

In [15]:
print("=== Reclasificacion al pasar de umbral unisex a umbral por sexo ===")
comparacion = alt_threshold_comparison(df)
display(comparacion)

print("\n=== Zona gris: normal con umbral unisex, elevado con umbral por sexo ===")
zona_gris = {}
for sexo in ("Female", "Male"):
    grupo = df[df["Gender"] == sexo]
    uln = ALT_ULN_BY_SEX[sexo]
    en_zona = grupo[(grupo["Sgpt"] > uln) & (grupo["Sgpt"] <= ALT_ULN_UNISEX)]
    zona_gris[sexo] = en_zona
    print(f"{sexo:>7}: {len(en_zona):>3} de {len(grupo):>3} personas "
          f"({100 * len(en_zona) / len(grupo):.1f}%) con {uln} < ALT <= {ALT_ULN_UNISEX}")
    print(f"{'':>7}  de esas, etiquetadas NO hepaticas (Selector=2): "
          f"{int((en_zona['Selector'] == 2).sum())}")

print("\n=== Ancho de la ventana en disputa (el caveat aritmetico) ===")
for sexo in ("Female", "Male"):
    ancho = ALT_ULN_UNISEX - ALT_ULN_BY_SEX[sexo]
    print(f"{sexo:>7}: ventana de {ALT_ULN_BY_SEX[sexo]} a {ALT_ULN_UNISEX} = {ancho} U/L de ancho")

print("\n=== Etiquetados 'no hepaticos' con ALT por encima de SU ULN ===")
for sexo in ("Female", "Male"):
    sanos = df[(df["Gender"] == sexo) & (df["Selector"] == 2)]
    uln = ALT_ULN_BY_SEX[sexo]
    n_sexo = int((sanos["Sgpt"] > uln).sum())
    n_uni = int((sanos["Sgpt"] > ALT_ULN_UNISEX).sum())
    print(f"{sexo:>7} (n={len(sanos):>3}): ALT>{uln:>2} (por sexo) -> {n_sexo:>3} "
          f"({100 * n_sexo / len(sanos):.1f}%)  |  ALT>{ALT_ULN_UNISEX} (unisex) -> {n_uni:>3} "
          f"({100 * n_uni / len(sanos):.1f}%)")

=== Reclasificacion al pasar de umbral unisex a umbral por sexo ===


,n,ULN_sexo,anormales_unisex,pct_unisex,anormales_por_sexo,pct_por_sexo,reclasificados,delta_pp
Gender,,,,,,,,
Female,142,19,39,27.5,109,76.8,70,49.3
Male,441,30,208,47.2,277,62.8,69,15.6



=== Zona gris: normal con umbral unisex, elevado con umbral por sexo ===
 Female:  70 de 142 personas (49.3%) con 19 < ALT <= 40
         de esas, etiquetadas NO hepaticas (Selector=2): 27
   Male:  69 de 441 personas (15.6%) con 30 < ALT <= 40
         de esas, etiquetadas NO hepaticas (Selector=2): 18

=== Ancho de la ventana en disputa (el caveat aritmetico) ===
 Female: ventana de 19 a 40 = 21 U/L de ancho
   Male: ventana de 30 a 40 = 10 U/L de ancho

=== Etiquetados 'no hepaticos' con ALT por encima de SU ULN ===
 Female (n= 50): ALT>19 (por sexo) ->  34 (68.0%)  |  ALT>40 (unisex) ->   7 (14.0%)
   Male (n=117): ALT>30 (por sexo) ->  48 (41.0%)  |  ALT>40 (unisex) ->  30 (25.6%)


In [16]:
fig, (ax_a, ax_b) = plt.subplots(1, 2, figsize=(13.5, 5.6))

# --- Panel A: que porcion de cada sexo cae en cada banda de ALT ---------------
# Encoding ORDINAL (bandas con orden natural) -> rampa de un solo tono.
bandas = [
    (f"\u2264 {ALT_ULN_BY_SEX['Female']}", 0, ALT_ULN_BY_SEX["Female"]),
    (f"{ALT_ULN_BY_SEX['Female']}\u2013{ALT_ULN_BY_SEX['Male']}", ALT_ULN_BY_SEX["Female"], ALT_ULN_BY_SEX["Male"]),
    (f"{ALT_ULN_BY_SEX['Male']}\u2013{ALT_ULN_UNISEX}", ALT_ULN_BY_SEX["Male"], ALT_ULN_UNISEX),
    (f"> {ALT_ULN_UNISEX}", ALT_ULN_UNISEX, float("inf")),
]

sexos = ["Male", "Female"]  # Female arriba al invertir el eje
acumulado = {s: 0.0 for s in sexos}

for (etiqueta, lo, hi), color in zip(bandas, ORDINAL_RAMP_4):
    for y, sexo in enumerate(sexos):
        grupo = df[df["Gender"] == sexo]
        pct = 100 * ((grupo["Sgpt"] > lo) & (grupo["Sgpt"] <= hi)).mean()
        ax_a.barh(y, pct, left=acumulado[sexo], color=color, height=0.55,
                  edgecolor="#fcfcfb", linewidth=2)
        if pct >= 6:
            ax_a.text(acumulado[sexo] + pct / 2, y, f"{pct:.1f}%", ha="center",
                      va="center", fontsize=9.5, fontweight="bold", color="white")
        acumulado[sexo] += pct

# Corchete sobre la ZONA EN DISPUTA de cada sexo (entre su ULN y el unisex)
for y, sexo in enumerate(sexos):
    grupo = df[df["Gender"] == sexo]
    inicio = 100 * (grupo["Sgpt"] <= ALT_ULN_BY_SEX[sexo]).mean()
    ancho = 100 * ((grupo["Sgpt"] > ALT_ULN_BY_SEX[sexo]) & (grupo["Sgpt"] <= ALT_ULN_UNISEX)).mean()
    offset = 0.42 if sexo == "Female" else -0.42
    ax_a.plot([inicio, inicio + ancho], [y + offset] * 2, color=INK_PRIMARY, linewidth=2.2)
    ax_a.text(inicio + ancho / 2, y + offset * 1.42,
              f"zona en disputa: {ancho:.1f}%", ha="center",
              va="bottom" if sexo == "Female" else "top",
              fontsize=9, fontweight="bold", color=INK_PRIMARY)

ax_a.set_yticks(range(len(sexos)))
ax_a.set_yticklabels([f"{s}\n(n={int((df['Gender'] == s).sum())})" for s in sexos])
ax_a.set_xlim(0, 100)
ax_a.set_ylim(-0.85, len(sexos) - 0.15)
ax_a.set_xlabel("% del sexo en cada banda de ALT (U/L)")
ax_a.set_title("A \u00b7 D\u00f3nde vive cada sexo respecto de los umbrales\nLa mitad de las mujeres cae en la franja que el umbral decide",
               fontsize=10, pad=10)
ax_a.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=c) for c in ORDINAL_RAMP_4],
            labels=[b[0] for b in bandas], title="ALT (U/L)", frameon=False,
            ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.14), fontsize=9)
ax_a.grid(axis="x", color=GRID_COLOR)
ax_a.grid(axis="y", visible=False)

# --- Panel B: % clasificado anormal bajo cada criterio ------------------------
plot_data = pd.DataFrame([
    {"criterio": "Umbral unisex\n(40 U/L)", "Gender": s, "pct": comparacion.loc[s, "pct_unisex"]}
    for s in ("Female", "Male")
] + [
    {"criterio": "Umbral por sexo\n(19 M \u00b7 30 H)", "Gender": s, "pct": comparacion.loc[s, "pct_por_sexo"]}
    for s in ("Female", "Male")
])

sns.barplot(
    data=plot_data, x="criterio", y="pct", hue="Gender",
    hue_order=["Female", "Male"], palette=SEX_COLORS, ax=ax_b,
    edgecolor="#fcfcfb", linewidth=2, saturation=1, width=0.7,
)

for contenedor in ax_b.containers:
    ax_b.bar_label(contenedor, fmt="%.1f%%", fontsize=10,
                   fontweight="bold", color=INK_SECONDARY, padding=3)

ax_b.set_ylim(0, 100)
ax_b.set_xlabel("")
ax_b.set_ylabel("% del sexo clasificado como ALT anormal")
ax_b.set_title("B \u00b7 Mismas personas, distinto criterio\nLas mujeres pasan de estar por debajo a estar por encima",
               fontsize=10, pad=10)
ax_b.legend(title="", frameon=False, loc="upper left")
ax_b.grid(axis="y", color=GRID_COLOR)
ax_b.grid(axis="x", visible=False)

fig.suptitle("Sensibilidad de la clasificaci\u00f3n de ALT al umbral elegido (n=583)",
             fontsize=12.5, fontweight="bold")
fig.tight_layout()
save_figure(fig, "loopc_umbral_alt_por_sexo.png")
plt.show()

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_44988\3933855446.py:83: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Interpretación — el resultado y su caveat, en ese orden.**

### El resultado

| | Anormales con **umbral unisex (40)** | Anormales con **umbral por sexo** | Cambio |
|---|---|---|---|
| **Mujeres** (n=142) | 39 (27.5%) | 109 (**76.8%**) | **+49.3 pp** |
| **Hombres** (n=441) | 208 (47.2%) | 277 (**62.8%**) | +15.6 pp |

Cambiar el criterio mueve a **la mitad de las mujeres** (70 personas) y a **una sexta parte de los hombres** (69 personas). El impacto es **~3 veces mayor** sobre ellas.

El panel B muestra el efecto más llamativo: bajo el umbral unisex las mujeres aparecen **menos afectadas** que los hombres (27.5% vs 47.2%); bajo el umbral por sexo aparecen **más afectadas** (76.8% vs 62.8%). **La ordenación entre sexos se invierte** solo por cambiar la regla de medida.

Y entre los etiquetados como **no hepáticos**: el **68.0%** de las mujeres "sanas" tiene ALT por encima de su límite femenino, frente al **41.0%** de los hombres respecto del suyo.

### El caveat — parte del efecto es aritmético, y hay que decirlo

La ventana en disputa **no mide lo mismo en ambos sexos**:

- Mujeres: de 19 a 40 → **21 U/L** de ancho
- Hombres: de 30 a 40 → **10 U/L** de ancho

Una ventana más ancha atrapa más gente. **Era esperable** que se reclasificaran más mujeres, y sería deshonesto presentar los 49.3 pp como si todo el efecto fuera sustantivo.

**Lo que NO es aritmético:** una ventana ancha solo atrapa gente **si hay gente parada dentro**. Y **70 de las 142 mujeres (49.3%) tienen su ALT efectivamente dentro de esa franja**. Si la distribución femenina estuviera desplazada hacia valores altos, la ventana estaría vacía por ancha que fuera. El panel A lo muestra: **la masa de la distribución femenina se concentra justo entre las dos líneas**.

Esa concentración es un hecho de los datos, no del cálculo.

### Qué se puede afirmar y qué no

**No se puede afirmar:**

- ❌ Que los clínicos del ILPD usaran el umbral de 40 — el dataset no documenta sus criterios diagnósticos.
- ❌ Que esas 70 mujeres estén mal diagnosticadas — no hay *gold standard* contra el cual verificar.
- ❌ Que esto explique **toda** la brecha de 8.7 pp.

**Sí se puede afirmar, y es lo que se reporta:**

- ✅ La población femenina del ILPD está **desproporcionadamente concentrada en el rango de ALT donde la elección del umbral decide el resultado**.
- ✅ La elección de umbral tiene **~3 veces más impacto** sobre las mujeres que sobre los hombres en este dataset.
- ✅ Es un **mecanismo plausible** que contribuiría a la brecha observada, coherente con la literatura.

Esto es un **análisis de sensibilidad**: no prueba una causa, mide cuánto depende un resultado de una decisión metodológica que alguien tomó.

### Conexión con la literatura

**Straw & Wu (2022)** auditaron modelos entrenados sobre este mismo dataset y encontraron una tasa de falsos negativos consistentemente peor en mujeres — hasta **−24.07 pp** en regresión logística, −21.02 pp en Random Forest, −19.31 pp en Naïve Bayes. Al rebalancear por sexo, **`ALP` y `sexo` pasan a ser las dos variables más importantes** del modelo.

Y citan explícitamente el mecanismo de los umbrales (vía Suthahar et al.): *"lo que se considera 'normal' en un sexo puede no serlo en el otro"*, y *"la expresión más leve del daño hepático en mujeres puede hacer que la enfermedad femenina pase desapercibida"*.

**El aporte de esta sección** es pasar de esa afirmación cualitativa a una **medición sobre los datos concretos**.

### Limitación declarada — los rangos de referencia no son locales

Los umbrales de Prati proceden de **donantes de sangre en Milán**; el ILPD, de **Andhra Pradesh**. El estudio multicéntrico C-RIDL/IFCC en India (Shah et al. 2018, n=512, 4 ciudades) **sí confirma** que ALT, AST, ALP, albúmina y bilirrubina total requieren partición por sexo. Pero los estudios regionales indios **se contradicen entre sí** —un estudio de Mumbai encuentra ALT 36.0 (H) vs 23.6 (M), mientras dos de Karnataka **no** hallan diferencia significativa en ALT— y **ninguno cubre Andhra Pradesh**.

**No existe un rango de referencia consolidado aplicable con certeza a esta población.** Se reporta como limitación explícita, no se disimula: la dirección del hallazgo (ALT normal más bajo en mujeres) es robusta en poblaciones occidentales y asiáticas, pero **la magnitud exacta del umbral es incierta para esta cohorte**.

## Valor agregado — Cociente De Ritis: ¿una etiqueta, cuántas enfermedades?

### El razonamiento

`Selector` es una etiqueta **binaria**: hepático / no hepático. Pero el dataset **no registra la etiología** — no dice si el daño viene de alcohol, hepatitis viral, esteatosis u obstrucción (ambigüedad A1 del PRD). La pregunta es si bajo esa única etiqueta conviven poblaciones distintas.

En T5 se vio que `Sgpt` (ALT) y `Sgot` (AST) correlacionan a 0.79 — alto, pero **no 0.99**. Esa discrepancia no es ruido: **es información**. Las dos enzimas se derraman juntas cuando el hepatocito se rompe, pero en proporciones que dependen del tipo de daño.

El **cociente De Ritis** (`AST/ALT`, descrito en 1957) captura esa proporción:

| Cociente | Patrón sugerido | Mecanismo |
|---|---|---|
| **> 2** | Tipo alcohólico / cirrótico | El alcohol reduce la actividad de ALT; el daño mitocondrial libera AST de vida media más larga |
| **1 – 2** | Zona de transición | — |
| **< 1** | Tipo viral / esteatosis temprana | ALT predomina, sin daño mitocondrial marcado |

### ⚠️ Los límites de este argumento — se declaran antes de mirar los datos

La literatura es explícita en que este cociente **no es prueba etiológica**:

1. **No es específico de la causa.** Un cociente >2 puede reflejar alcohol, pero también cirrosis avanzada de *cualquier* origen, carcinoma hepatocelular, o daño muscular concomitante (AST no es exclusiva del hígado).
2. **Depende del estadio, no solo de la causa.** Un paciente con hepatitis viral sin fibrosis tiene cociente <1; **ese mismo paciente**, al desarrollar cirrosis, puede superar 2 sin cambiar de etiología. La distribución observada podría reflejar heterogeneidad de **estadio**, no de causa.
3. **Depende del método analítico.** Con ensayos IFCC estandarizados, el poder discriminativo del cociente cae drásticamente. No sabemos qué método usó el laboratorio del ILPD.
4. **Sin *gold standard*, no hay validación.** Las cifras de sensibilidad/especificidad publicadas vienen de estudios con etiología confirmada por biopsia o serología. Aquí no la hay.
5. **Riesgo de circularidad.** Usar el mismo cociente que la literatura emplea para *inferir* etiología, y luego presentarlo como *confirmación* de subgrupos etiológicos, es razonamiento circular.

**Por eso la conclusión se formula como *"sugiere plausibilidad de"*, nunca como *"demuestra"*.**

> Fuente: `docs/fuentes/Consulta_3.md`.

In [17]:
df_ritis = df.assign(de_ritis=de_ritis_ratio(df))
pacientes = df_ritis[df_ritis["Selector"] == 1]

print(f"Pacientes (Selector=1) con cociente calculable: {pacientes['de_ritis'].notna().sum()}")
print(f"Mediana global: {pacientes['de_ritis'].median():.2f}\n")

print("--- Distribucion del cociente en pacientes (Selector=1) ---")
tramos = {
    f"> {DE_RITIS_ALCOHOLIC} (tipo alcoholico/cirrotico)": pacientes["de_ritis"] > DE_RITIS_ALCOHOLIC,
    f"{DE_RITIS_VIRAL} - {DE_RITIS_ALCOHOLIC} (transicion)": pacientes["de_ritis"].between(DE_RITIS_VIRAL, DE_RITIS_ALCOHOLIC),
    f"< {DE_RITIS_VIRAL} (tipo viral/esteatosis)": pacientes["de_ritis"] < DE_RITIS_VIRAL,
}
for etiqueta, mascara in tramos.items():
    n = int(mascara.sum())
    print(f"  {etiqueta:<45} {n:>3} ({100 * n / len(pacientes):.1f}%)")

print("\n--- Mismo corte, estratificado por sexo ---")
# Consulta_3 advierte que el cociente 'sano' difiere por sexo: se verifica.
resumen_sexo = []
for sexo in ("Female", "Male"):
    grupo = pacientes[pacientes["Gender"] == sexo]["de_ritis"].dropna()
    resumen_sexo.append({
        "Gender": sexo, "n": len(grupo), "mediana": round(grupo.median(), 2),
        "pct_mayor_2": round(100 * (grupo > DE_RITIS_ALCOHOLIC).mean(), 1),
        "pct_menor_1": round(100 * (grupo < DE_RITIS_VIRAL).mean(), 1),
    })
display(pd.DataFrame(resumen_sexo).set_index("Gender"))

Pacientes (Selector=1) con cociente calculable: 416
Mediana global: 1.22

--- Distribucion del cociente en pacientes (Selector=1) ---
  > 2.0 (tipo alcoholico/cirrotico)              75 (18.0%)
  1.0 - 2.0 (transicion)                        190 (45.7%)
  < 1.0 (tipo viral/esteatosis)                 151 (36.3%)

--- Mismo corte, estratificado por sexo ---


,n,mediana,pct_mayor_2,pct_menor_1
Gender,,,,
Female,92,1.17,12.0,39.1
Male,324,1.25,19.8,35.5


In [18]:
RITIS_VIEW_MAX = 5  # recorte de vista; la cola larga aplastaria la zona de interes

fig, (ax_h, ax_v) = plt.subplots(1, 2, figsize=(13, 5.4))

# --- Panel A: histograma + KDE -> la forma responde "¿hay bimodalidad?" -------
datos_hist = pacientes["de_ritis"].dropna().clip(upper=RITIS_VIEW_MAX)
sns.histplot(datos_hist, bins=40, kde=True, ax=ax_h,
             color=SEX_COLORS["Female"], edgecolor="#fcfcfb", linewidth=0.5)

for corte, texto in [(DE_RITIS_VIRAL, f"{DE_RITIS_VIRAL}"), (DE_RITIS_ALCOHOLIC, f"{DE_RITIS_ALCOHOLIC}")]:
    ax_h.axvline(corte, color=INK_SECONDARY, linestyle="--", linewidth=1.4)
    ax_h.text(corte, ax_h.get_ylim()[1] * 0.96, f" {texto}", fontsize=9,
              fontweight="bold", color=INK_SECONDARY, va="top")

ax_h.text(0.5, ax_h.get_ylim()[1] * 0.55, "tipo viral /\nesteatosis", ha="center",
          fontsize=8.5, color=INK_SECONDARY)
ax_h.text(3.4, ax_h.get_ylim()[1] * 0.55, "tipo alcohólico /\ncirrótico", ha="center",
          fontsize=8.5, color=INK_SECONDARY)

ax_h.set_xlim(0, RITIS_VIEW_MAX)
ax_h.set_xlabel("Cociente De Ritis (AST / ALT)")
ax_h.set_ylabel("Nº de pacientes")
ax_h.set_title("A · Forma de la distribución (Selector = 1, n=416)\n¿Un solo pico o varios?",
               fontsize=10, pad=10)
ax_h.grid(axis="y", color=GRID_COLOR)
ax_h.grid(axis="x", visible=False)

# --- Panel B: violin -> comparacion entre los 4 grupos Gender x Selector ------
df_violin = df_ritis.dropna(subset=["de_ritis"]).copy()
df_violin["grupo"] = df_violin["Selector"].map({1: "Hepático", 2: "No hepático"})
df_violin["de_ritis_vista"] = df_violin["de_ritis"].clip(upper=RITIS_VIEW_MAX)

sns.violinplot(
    data=df_violin, x="grupo", y="de_ritis_vista", hue="Gender",
    hue_order=["Female", "Male"], order=["Hepático", "No hepático"],
    palette=SEX_COLORS, split=True, gap=0.12, inner="quartile",
    linewidth=1.1, saturation=1, cut=0, ax=ax_v,
)

for corte in (DE_RITIS_VIRAL, DE_RITIS_ALCOHOLIC):
    ax_v.axhline(corte, color=INK_SECONDARY, linestyle="--", linewidth=1.2, zorder=0)

ax_v.set_ylim(0, RITIS_VIEW_MAX)
ax_v.set_xlabel("")
ax_v.set_ylabel("Cociente De Ritis (AST / ALT)")
ax_v.set_title("B · Comparación por sexo y etiqueta\nLos cuatro grupos se solapan ampliamente",
               fontsize=10, pad=10)
ax_v.legend(title="", frameon=False, loc="upper right")
ax_v.grid(axis="y", color=GRID_COLOR)
ax_v.grid(axis="x", visible=False)

fig.suptitle(f"Cociente De Ritis — vista recortada en {RITIS_VIEW_MAX} para legibilidad",
             fontsize=11.5, fontweight="bold")
fig.tight_layout()
save_figure(fig, "loopc_de_ritis.png")
plt.show()

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_44988\1271646202.py:56: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Interpretación — formulada como hipótesis, no como conclusión.**

### Lo observado

Sobre los 416 pacientes etiquetados como hepáticos, mediana **1.22**:

| Tramo | n | % | Patrón sugerido |
|---|---|---|---|
| `> 2` | 75 | **18.0%** | Tipo alcohólico / cirrótico |
| `1 – 2` | 190 | 45.7% | Transición |
| `< 1` | 151 | **36.3%** | Tipo viral / esteatosis |

Más de un tercio de los pacientes está por debajo de 1 y casi una quinta parte por encima de 2 — **los dos extremos del espectro están sustancialmente poblados**, con la mediana en la zona de transición.

### Lo que esto sugiere (y lo que no)

**Es compatible** con que la etiqueta binaria `Selector = 1` agrupe pacientes cuyo daño hepático se manifiesta de formas bioquímicamente distintas. Si toda la cohorte compartiera un mismo mecanismo y estadio, cabría esperar una distribución más concentrada.

**Pero, honestamente:** el panel A **no muestra bimodalidad clara**. La distribución es unimodal con cola derecha, no dos jorobas separadas. Una mezcla de subpoblaciones *puede* producir esta forma si los grupos se solapan mucho, pero **la forma por sí sola no permite distinguir "mezcla de poblaciones solapadas" de "una sola población dispersa"**.

Y como se declaró antes de mirar los datos: **el cociente refleja también el estadio de la enfermedad**. Un 18% con cociente >2 es igualmente compatible con "18% de pacientes alcohólicos" que con "18% de pacientes con cirrosis avanzada de cualquier causa". **Sin la etiología registrada, ambas lecturas son indistinguibles.**

### Formulación adoptada para el informe

> *"La distribución del cociente De Ritis en la cohorte hepática **sugiere la plausibilidad** de heterogeneidad no capturada por la etiqueta binaria — sea etiológica, de estadio, o ambas. La ausencia de una variable de etiología en el dataset (ambigüedad A1) impide discriminar entre esas explicaciones."*

Esto es una **hipótesis generadora**, no un resultado. Su valor está en documentar una limitación real del dataset con evidencia cuantitativa, no en clasificar pacientes.

### Diferencia por sexo — modesta pero en la dirección esperada

| | n | Mediana | `> 2` | `< 1` |
|---|---|---|---|---|
| **Mujeres** | 92 | **1.17** | 12.0% | 39.1% |
| **Hombres** | 324 | **1.25** | **19.8%** | 35.5% |

Los hombres presentan patrón "tipo alcohólico" con **casi el doble de frecuencia** (19.8% vs 12.0%). Es coherente con la epidemiología conocida de la hepatopatía alcohólica, **pero el dataset no registra consumo de alcohol** — así que es una asociación sugerente, no una explicación verificada.

⚠️ La `Consulta_3` advierte además que el cociente **"sano" difiere por sexo** (referencias de hasta 1.7 en mujeres vs 1.3 en hombres). Es decir: **parte de esta diferencia podría ser fisiología basal, no patología**. Un análisis riguroso requeriría umbrales de De Ritis específicos por sexo, que la literatura revisada **no proporciona de forma consolidada**. Se declara como vacío de evidencia.

El panel B lo confirma visualmente: **los cuatro grupos se solapan ampliamente**. Ni el sexo ni la etiqueta separan la distribución del cociente de forma nítida — lo cual es en sí mismo el hallazgo honesto.

### Aporte al proyecto

Esta sección **no descubre subgrupos**. Documenta, con evidencia cuantitativa y sus limitaciones explícitas, que **la etiqueta `Selector` es más gruesa que la realidad clínica que pretende representar** — reforzando el argumento de *proxy label* de la Fase 1 y la ambigüedad A1 del PRD.

## Notas para fases futuras (no se ejecuta nada aquí)

### Fase 3 — Preprocessing · mandatos derivados de este notebook

**T6 — Imputación**

- Comparar honestamente imputación por media vs. determinista: el error de reconstrucción de `A/G Ratio` es **0.051 en promedio, con 33.0% de filas por encima de 0.05**. Citarlo tal cual; **no** redondear a "la fórmula es exacta".
- **Las 3 filas con `DB > TB` (Q8):** marcar **ambas** columnas `TB` y `DB` como faltantes en esas filas, **sin eliminar la fila**. Justificación: (a) no hay evidencia de cuál valor es el erróneo, así que "corregir" sería inventar; (b) el resto de la analítica de esos 3 pacientes es válida. Documentar como violación de plausibilidad según el marco de Kahn et al. (2016) / OHDSI.
- ⚠️ **Efecto sobre T3:** anular esas 3 filas cambia levemente los estadísticos de `TB` y `DB`. **T3 se sigue reportando sobre los datos originales** (regla del PRD: no mezclar estadísticos descriptivos con datos ya tratados); añadir una nota de sensibilidad indicando la magnitud del cambio.

**T7 — Escalado**

- La evidencia de T4 anticipa la conclusión: con `Sgot` (max 4929, mediana 42) y `Sgpt` (max 2000), MinMax comprimirá las medianas contra el cero. Cuantificarlo en F3-R9.
- Recordar que el eje de síntesis (`TP`, `ALB`) es casi simétrico: el efecto del escalado **no es uniforme entre variables**.

**T8 — Outliers**

- **[M] Obligatorio:** Tukey global sobre las 583 filas, sin excepciones. No se sustituye.
- **[V] Valor agregado:** repetir estratificado por **franja de edad**, citando CLSI C28-A3 y CALIPER, y reportar cuántos menores son marcados como atípicos por el umbral global pese a estar dentro de su rango pediátrico normal (129–468 U/L según edad y sexo).
- **[V] F3-R15:** repetir estratificado por **sexo**. Los rangos pediátricos de ALP **también difieren por sexo**, así que un tratamiento ciego a ambas variables acumula dos sesgos sobre el mismo subgrupo.
- Mantener la distinción del §9.2 del PRD: **outlier estadístico** (lejos del centro pero clínicamente plausible → conservar) vs **outlier erróneo** (valor imposible → corregir). Las 3 filas `DB > TB` son del segundo tipo; los `Sgot` extremos, del primero.

**T2 — Duplicados**

- Decidir el tratamiento de los **13 duplicados exactos**; no se eliminan en este notebook para no alterar los números reportados en T1–T5.

### Fase E — Entregables

- El informe debe declarar como limitaciones explícitas: (a) **no existe rango de referencia validado para Andhra Pradesh**, los estudios regionales indios se contradicen; (b) `Age` tiene **doble problema de calidad** (heaping tosco + top-coding en 90), por lo que no admite análisis etario fino; (c) el dataset **carece de variable de etiología**, y el cociente De Ritis solo permite hipótesis, no clasificación.
- La metodología de investigación bibliográfica está versionada en `docs/fuentes/` — cuatro consultas de investigación profunda cuyas respuestas fundamentan cada umbral y criterio citado.

### Fase 4+ — Modelado (fuera del alcance de este PRD)

- `Selector` debe recodificarse (1/2 → 1/0) antes de entrenar.
- El par `Sgpt`/`Sgot` requiere evaluación de **VIF** antes de descartar alguna — Straw & Wu muestran que pesan distinto por sexo.
- El *split* train/test debe ser sustancialmente mayor a 29 filas y estratificado por `Selector` **y** `Gender` (ver `docs/adr/0003-sin-holdout-en-fases-0-3.md`).
- **Hipótesis para la auditoría de *fairness* (Fase 5):** el análisis de sensibilidad de umbrales de este notebook cuantifica un mecanismo candidato para la brecha de 8.7 pp. Contrastar si un modelo entrenado con la etiqueta original reproduce la disparidad de FNR que reportan Straw & Wu (hasta −24.07 pp en regresión logística).
- ⚠️ El escalado y la imputación deberán ir **dentro de un `Pipeline` ajustado solo con el train set**. Ajustar con todo el dataset filtra información del test e infla optimistamente las métricas.